# Text-only Emotion Baseline (RoBERTa)

MELD 대사의 텍스트만 사용해 7개 감정을 분류합니다.

## 1. 환경 설정과 재현성

In [ ]:
# 필요한 라이브러리를 불러옵니다.
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm


In [2]:
# 재현 가능한 결과를 위해 난수 시드를 고정합니다.
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## 2. 라벨과 데이터 준비

In [3]:
# MELD의 7개 감정 라벨과 정수 ID의 대응표를 만듭니다.
emotions = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise",
]

label2id = {
    label: idx
    for idx, label in enumerate(emotions)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print(label2id)

{'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}


In [7]:
# train/dev/test 메타데이터 CSV를 읽습니다.
train_df = pd.read_csv("/workspace/emotion-ai/data/meld_metadata/train_sent_emo.csv")
dev_df = pd.read_csv("/workspace/emotion-ai/data/meld_metadata/dev_sent_emo.csv")
test_df = pd.read_csv("/workspace/emotion-ai/data/meld_metadata/test_sent_emo.csv")

In [8]:
# 오디오와 공정하게 비교할 수 있도록 손상된 오디오 2개를 test에서 제외합니다.
bad_audio_samples = [
    (38, 4),
    (220, 0),
]

common_test_df = test_df[
    ~test_df.apply(
        lambda row: (
            row["Dialogue_ID"],
            row["Utterance_ID"]
        ) in bad_audio_samples,
        axis=1
    )
].reset_index(drop=True)

print(len(common_test_df))
# 2608

2608


## 3. Tokenizer

In [4]:
# 사전학습된 RoBERTa 토크나이저를 불러옵니다.
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## 4. Dataset과 DataLoader

In [6]:
# 문장 토큰화와 라벨 변환을 담당하는 PyTorch Dataset을 정의합니다.
class MELDTextDataset(Dataset):
    def __init__(
        self,
        dataframe,
        tokenizer,
        label2id,
        max_length=128
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        text = str(row["Utterance"])
        label = self.label2id[row["Emotion"]]

        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [9]:
# 각 데이터 분할을 MELDTextDataset 객체로 만듭니다.
train_dataset = MELDTextDataset(
    train_df,
    tokenizer,
    label2id,
    max_length=128,
)

dev_dataset = MELDTextDataset(
    dev_df,
    tokenizer,
    label2id,
    max_length=128,
)

test_dataset = MELDTextDataset(
    common_test_df,
    tokenizer,
    label2id,
    max_length=128,
)

In [11]:
# 학습·검증·테스트용 DataLoader를 구성합니다.
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

## 5. Modeling

RoBERTa의 첫 토큰 표현을 문장 임베딩

In [13]:
# RoBERTa의 첫 토큰 표현을 분류하는 텍스트 감정 모델을 정의합니다.
class TextEmotionClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        num_labels,
        dropout=0.1
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state

        sentence_embedding = hidden_states[:, 0, :]

        sentence_embedding = self.dropout(
            sentence_embedding
        )

        logits = self.classifier(
            sentence_embedding
        )

        return logits

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TextEmotionClassifier(
    model_name=MODEL_NAME,
    num_labels=len(emotions),
)

model = model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Loss, Optimizer, 학습 함수

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
)


In [ ]:
def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device
):
    model.train()

    total_loss = 0

    for batch in tqdm(dataloader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
@torch.no_grad()
def evaluate(
    model,
    dataloader,
    criterion,
    device
):
    model.eval()

    total_loss = 0

    all_preds = []
    all_labels = []

    for batch in tqdm(dataloader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        loss = criterion(
            logits,
            labels
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        total_loss += loss.item()

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
    )

    return (
        total_loss / len(dataloader),
        macro_f1,
        all_labels,
        all_preds,
    )

## 7. Main Training

3 epoch

In [ ]:
# 전체 학습 데이터를 3 epoch 학습
NUM_EPOCHS = 3

for epoch in range(NUM_EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
    )

    dev_loss, dev_f1, _, _ = evaluate(
        model,
        dev_loader,
        criterion,
        device,
    )

    print(
        f"Epoch {epoch + 1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Dev Loss: {dev_loss:.4f} | "
        f"Dev Macro F1: {dev_f1:.4f}"
    )

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 1.2259 | Dev Loss: 1.1981 | Dev Macro F1: 0.3927


  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 1.0263 | Dev Loss: 1.1616 | Dev Macro F1: 0.4124


  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

Epoch 3 | Train Loss: 0.8622 | Dev Loss: 1.1798 | Dev Macro F1: 0.4420


## 8. Test 평가와 예측 저장

In [25]:
# 학습된 모델을 test set에서 평가하고 클래스별 지표를 출력합니다.
test_loss, test_f1, test_labels, test_preds = evaluate(
    model,
    test_loader,
    criterion,
    device,
)

print("Test Macro F1:", test_f1)

print(
    classification_report(
        test_labels,
        test_preds,
        target_names=emotions,
        digits=4,
    )
)

  0%|          | 0/163 [00:00<?, ?it/s]

Test Macro F1: 0.4530138454291022
              precision    recall  f1-score   support

       anger     0.4759    0.4870    0.4814       345
     disgust     0.2245    0.1642    0.1897        67
        fear     0.2727    0.1200    0.1667        50
         joy     0.5777    0.6584    0.6154       401
     neutral     0.7729    0.7994    0.7859      1256
     sadness     0.4857    0.3269    0.3908       208
    surprise     0.5347    0.5480    0.5413       281

    accuracy                         0.6423      2608
   macro avg     0.4777    0.4434    0.4530      2608
weighted avg     0.6314    0.6423    0.6343      2608



In [ ]:
text_result = common_test_df[
    [
        "Dialogue_ID",
        "Utterance_ID",
        "Utterance",
        "Emotion",
    ]
].copy()

text_result["true_id"] = test_labels
text_result["text_pred_id"] = test_preds

results_dir = Path("/workspace/emotion-ai/results")
results_dir.mkdir(parents=True, exist_ok=True)

text_result.to_csv(
    results_dir / "text_test_predictions.csv",
    index=False,
)


## 9. 결과 및 오류 분석

In [ ]:
# test 예측의 confusion matrix를 계산합니다.
cm = confusion_matrix(test_labels, test_preds)

cm_df = pd.DataFrame(
    cm,
    index=emotions,
    columns=emotions,
)

print(cm_df)


In [33]:
# 클래스별 비율로 정규화한 confusion matrix를 확인합니다.
cm_norm = confusion_matrix(
    test_labels,
    test_preds,
    normalize="true"
)

cm_norm_df = pd.DataFrame(
    cm_norm,
    index=emotions,
    columns=emotions
)

print(cm_norm_df.round(3))

          anger  disgust   fear    joy  neutral  sadness  surprise
anger     0.487    0.043  0.009  0.142    0.191    0.023     0.104
disgust   0.294    0.162  0.000  0.044    0.382    0.088     0.029
fear      0.220    0.040  0.120  0.020    0.360    0.120     0.120
joy       0.060    0.012  0.002  0.659    0.177    0.020     0.070
neutral   0.043    0.007  0.004  0.074    0.799    0.032     0.041
sadness   0.139    0.014  0.019  0.101    0.346    0.327     0.053
surprise  0.167    0.014  0.011  0.093    0.153    0.014     0.548


In [36]:
# 정답과 예측의 감정별 개수 분포를 비교합니다.
true_names = [id2label[x] for x in test_labels]
pred_names = [id2label[x] for x in test_preds]

print("True")
print(Counter(true_names))

print("\nPred")
print(Counter(pred_names))

True
Counter({'neutral': 1256, 'joy': 402, 'anger': 345, 'surprise': 281, 'sadness': 208, 'disgust': 68, 'fear': 50})

Pred
Counter({'neutral': 1300, 'joy': 458, 'anger': 353, 'surprise': 288, 'sadness': 140, 'disgust': 49, 'fear': 22})


In [ ]:
# 정답과 예측이 다른 test 문장만 모아 오류 분석 표를 만듭니다.
# common_test_df를 사용해야 제외한 오디오 2개와 예측 길이가 일치합니다.
error_df = common_test_df.copy()
error_df["true_emotion"] = [id2label[x] for x in test_labels]
error_df["pred_emotion"] = [id2label[x] for x in test_preds]

errors = error_df[
    error_df["true_emotion"] != error_df["pred_emotion"]
].copy()


In [39]:
# 관심 있는 오분류 조합에서 실제 문장 예시를 일부 출력합니다.
pairs = [
    ("disgust", "neutral"),
    ("fear", "neutral"),
    ("sadness", "neutral"),
]

for true_label, pred_label in pairs:

    subset = errors[
        (errors["true_emotion"] == true_label)
        & (errors["pred_emotion"] == pred_label)
    ]

    print(f"\n===== {true_label} → {pred_label} =====")

    for text in subset["Utterance"].head(10):
        print("-", text)

In [41]:
# 주요 오분류 조합의 개수와 문장 예시를 자세히 확인합니다.
target_errors = [
    ("fear", "neutral"),
    ("disgust", "neutral"),
    ("sadness", "neutral"),
]

for true_label, pred_label in target_errors:

    subset = errors[
        (errors["true_emotion"] == true_label)
        &
        (errors["pred_emotion"] == pred_label)
    ]

    print("\n" + "=" * 70)
    print(f"{true_label.upper()} → {pred_label.upper()}")
    print(f"Count: {len(subset)}")
    print("=" * 70)

    for i, row in subset.head(10).iterrows():
        print(f"- {row['Utterance']}")


FEAR → NEUTRAL
Count: 18
- Look, we have enough, just walk away.
- Umm, see, I was thinking maybe you two could switch apartments because Phoebe's more our kind of people.
- Noo!  Oh, it’s okay, it’s okay, baby, baby, baby.
- Where is Leslie?
- Where are you Leslie?
- Do you have a minute? I’d like to talk to you about something I’m, I’m really uncomfortable talking about.
- Are we okay now?
- Umm, about that-that-that last move where the woman tripped you and then pinned you to the floor, what-what-what-what would you do next?
- This actor friend and he would kill me if he thought I was doing this umm, but umm would it be possible for him to get an audition for your movie say on Thursday?
- But Ben is coming over tonight and he can’t see this.

DISGUST → NEUTRAL
Count: 26
- Four percent. Okay. I tip more than that when there’s a bug in my food.
- Yeah, I’m gonna go to a doctor who went to school in a mini-mall.
- No, I don’t.
- To tape the game? You do this every time Ross, you’re
- 

Tokenizer 출력 확인

In [ ]:
sample = tokenizer(
    "I can't believe you did that!",
    truncation=True,
    padding="max_length",
    max_length=32,
    return_tensors="pt",
)

print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)

print(sample["input_ids"])
print(sample["attention_mask"])

torch.Size([1, 32])
torch.Size([1, 32])
tensor([[  0, 100,  64,  75, 679,  47, 222,  14, 328,   2,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])


In [ ]:
text = "I love you"

encoded = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=16
)

print("input_ids:")
print(encoded["input_ids"])

print("\ntokens:")
print(
    tokenizer.convert_ids_to_tokens(
        encoded["input_ids"]
    )
)

print("\nattention_mask:")
print(encoded["attention_mask"])

input_ids:
[0, 100, 657, 47, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

tokens:
['<s>', 'I', 'Ġlove', 'Ġyou', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']

attention_mask:
[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
# Dataset 한 샘플의 키, 텐서 크기, 라벨을 확인
sample = train_dataset[0]

print(sample.keys())
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["labels"])
print(id2label[sample["labels"].item()])

dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([128])
torch.Size([128])
tensor(4)
neutral


In [ ]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16])
